# Querying the dashboard API

Example calls to every endpoint of the API in `src/api/`, the server the React dashboard will
use. Each call goes over HTTP exactly as the dashboard's will, and the answers come back as JSON,
turned here into tables and Plotly charts.

**Start the API first,** from the repo root (needs `pip install -e ".[api,stats]"`):

```bash
uvicorn src.api.app:app --reload
```

Or leave `START_SERVER_IF_NEEDED = True` below, and the notebook starts one inside this kernel
if nothing answers. The server's own docs page, with every endpoint and its options, is at
http://127.0.0.1:8000/docs.

Runs offline on the synthetic tickers (generated from code, not market data). Set
`TICKER_SET = 'local'` for the tickers you have saved (the dashboard's **Download all**), or
`'yahoo'` for the tickers in `configs/tickers.yaml`, downloaded from Yahoo Finance.
This notebook needs a local server, so it does not run on Colab.

| Section | Endpoints |
|---|---|
| 1 | `GET /api/tickers`: the ticker sets and their parameters |
| 2 | `POST /api/overview`: series, latest snapshot, distribution |
| 3 | `POST /api/streaks`, `POST /api/cumulative` |
| 4 | `POST /api/rare-events`: the rare-event table, refiltered live |
| 5 | `GET /api/charts`, `POST /api/charts/{name}` |
| 6 | `POST /api/statistics/{section}` |
| 7 | `GET /api/multi-ticker` |
| 8 | Errors: what a bad request gets back |
| 9 | The same numbers as the package |

In [ ]:
import json
import time
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import pandas as pd
import plotly.io as pio
from IPython.display import display

BASE = 'http://127.0.0.1:8000'
START_SERVER_IF_NEEDED = True   # start the API inside this kernel if nothing answers at BASE
TICKER_SET = 'synthetic'        # 'synthetic' (offline), 'local' (saved data) or 'yahoo' (live)
TICKER = 'SYN-INDEX'            # a symbol from that set; 'ES=F' for the Yahoo or saved set


class ApiError(Exception):
    """An HTTP error from the API, with its status code and the server's explanation."""
    def __init__(self, status, detail):
        super().__init__(f'{status}: {detail}')
        self.status, self.detail = status, detail


def call(method, path, body=None, **query):
    """One request; query values may be lists (`n_days=[3, 5]` sends n_days=3&n_days=5)."""
    url = BASE + path + ('?' + urlencode(query, doseq=True) if query else '')
    data = None if body is None else json.dumps(body).encode()
    request = Request(url, data=data, method=method, headers={'Content-Type': 'application/json'})
    try:
        with urlopen(request, timeout=600) as response:
            return json.load(response)
    except HTTPError as exc:
        raise ApiError(exc.code, json.load(exc).get('detail')) from None


def get(path, **query):
    return call('GET', path, **query)


def post(path, body, **query):
    return call('POST', path, body, **query)


def table(t):
    """An API table ({columns, records}) as a DataFrame, columns in the server's order."""
    return pd.DataFrame(t['records'], columns=t['columns'])


def chart(figure_json):
    """An API chart as a Plotly figure: call .show() on it."""
    return pio.from_json(json.dumps(figure_json))

In [ ]:
# Q: Is the API up? If not, start one in this kernel.
def api_version():
    try:
        return get('/api/health')['version']
    except (URLError, ConnectionError):
        return None

if api_version() is None and START_SERVER_IF_NEEDED:
    import threading
    import uvicorn
    from src.api.app import app

    server = uvicorn.Server(uvicorn.Config(app, host='127.0.0.1', port=int(BASE.rsplit(':', 1)[1]),
                                           log_level='warning'))
    threading.Thread(target=server.run, daemon=True).start()
    for _ in range(40):
        if api_version():
            break
        time.sleep(0.25)
    print('Started the API inside this kernel; it stops when the kernel does.')

version = api_version()
if version is None:
    raise RuntimeError(f'No API at {BASE}. Start it from the repo root: uvicorn src.api.app:app')
print(f'API {version} at {BASE}')

## 1. Tickers and parameters

Every `POST` endpoint takes the same body: the `Params` fields as JSON (thresholds, windows,
dates, data source). What to show goes in the query string. Taking the body from
`/api/tickers` gives each ticker the parameters its config sets.

In [ ]:
# Q: Which tickers does the set offer, and with which thresholds?
tickers = get('/api/tickers', set=TICKER_SET)
print(f"{len(tickers['tickers'])} tickers from {tickers['source']}")
pd.DataFrame([{'symbol': t['symbol'], 'label': t['label'],
               'data_source': t['params']['data_source'],
               'start_date': t['params']['start_date'],
               'win_threshold': t['params']['win_threshold'],
               'loss_threshold': t['params']['loss_threshold']} for t in tickers['tickers']])

In [ ]:
# The request body used below: TICKER's parameters from its config. Edit any field to see
# the answers change, e.g. P = {**P, 'win_threshold': 0.5, 'loss_threshold': -0.5}.
P = next(t['params'] for t in tickers['tickers'] if t['symbol'] == TICKER)
P

## 2. Overview

In [ ]:
# Q: What series did the server load, and where do return and risk sit now?
overview = post('/api/overview', P)
print(f"{overview['ticker']} ({overview['data_source']}): {overview['rows']} daily returns, "
      f"{overview['start'][:10]} to {overview['end'][:10]}, last price {overview['last_price']:,.2f}")
display(pd.Series(overview['snapshot'], name='latest snapshot').to_frame())
pd.DataFrame([overview['distribution']])

## 3. Streaks and cumulative moves

In [ ]:
# Q: How often do wins and losses come in runs, and how often does the compounded move over a
# window clear each threshold?
display(table(post('/api/streaks', P)['summary']))
table(post('/api/cumulative', P)['summary'])

## 4. Rare events

The server builds the full table once (every threshold × holding period × lookback × event
type) and caches it; each call returns the rows with `prob` between the body's `prob_min` and
`prob_max`. Moving those bounds only refilters, which is what the dashboard's controls will do.

In [ ]:
# Q: For a 3-day holding period, which moves are rare but did happen?
rare = post('/api/rare-events', P, n_days=3)
print(f"{len(rare['events']['records'])} of {rare['total_events']} events have "
      f"{rare['prob_min']:.2%} < prob < {rare['prob_max']:.0%}")
table(rare['events']).head(10)

In [ ]:
# Q: What is left with a tighter bound, and how fast is a refilter?
for prob_max in (0.10, 0.05, 0.01):
    start = time.perf_counter()
    events = post('/api/rare-events', {**P, 'prob_max': prob_max}, n_days=3)['events']
    print(f'prob_max {prob_max:>5.0%}: {len(events["records"]):>3} events '
          f'({(time.perf_counter() - start) * 1000:.0f} ms)')

In [ ]:
# Q: Only cumulative moves, over 1 and 5 days together?
table(post('/api/rare-events', P, n_days=[1, 5], change_type='cumulative')['events'])

## 5. Charts

Each chart is the same `viz.plot_*` figure the notebooks draw, sent as Plotly JSON. The
dashboard renders the same JSON with plotly.js.

In [ ]:
# Q: Which charts are there, and which query options does each read?
pd.DataFrame(get('/api/charts')['charts'])

In [ ]:
# Q: What do the series and its streaks look like?
chart(post('/api/charts/price-and-returns', P)).show()
chart(post('/api/charts/streak-timeline', P, window=3)).show()     # 3-day streaks
chart(post('/api/charts/rolling-volatility', P)).show()

In [ ]:
# Q: Every chart at once: pick any names from the table above.
SHOW = ['return-distribution', 'cumulative-heatmap', 'qq', 'drawdown']
for name in SHOW:
    chart(post(f'/api/charts/{name}', P)).show()

## 6. Statistics

The statistics notebook's four sections. `probabilities` runs a block bootstrap, so it takes a
few seconds the first time (`n_boot` resamples, at most 2,000) and is cached after that.

In [ ]:
# Q: How far from normal are the returns, and how fat are the tails?
dist = post('/api/statistics/distribution', P)
display(pd.concat({'moments': pd.Series(dist['moments']),
                   'Jarque-Bera': pd.Series(dist['jarque_bera'])[['statistic', 'p_value']],
                   'Student-t fit': pd.Series(dist['student_t']),
                   'Hill tail index': pd.Series(dist['tail_index'])}).to_frame('value'))
var = table(dist['value_at_risk'])
var.pivot_table(index=['horizon', 'level'], columns='method', values=['var', 'es']).map('{:.2%}'.format)

In [ ]:
# Q: Is there memory in direction or in size, and does volatility cluster?
dep = post('/api/statistics/dependence', P)
display(table(dep['ljung_box']))
display(table(dep['variance_ratio']))
pd.Series(dep['arch_lm'], name='ARCH-LM').to_frame()

In [ ]:
# Q: How good was the return for the risk, and how deep were the worst drawdowns?
dd = post('/api/statistics/drawdowns', P, top=5)
display(pd.Series(dd['risk_ratios'], name='value').to_frame())
table(dd['drawdowns'])

In [ ]:
# Q: How precise are the rare-event probabilities, and what do i.i.d. models predict?
N_DAYS, N_BOOT = 3, 1000
start = time.perf_counter()
probs = post('/api/statistics/probabilities', P, n_days=N_DAYS, n_boot=N_BOOT)
events = table(probs['events'])
print(f'{len(events)} events, {time.perf_counter() - start:.1f} s')
rare = events[(events['prob'] < P['prob_max']) & (events['prob'] > P['prob_min'])]
rare.sort_values(['n_years', 'change_type', 'change', 'threshold'])

In [ ]:
for change in ('above', 'below'):
    chart(post('/api/charts/event-probabilities', P, n_days=N_DAYS, n_boot=N_BOOT,
               change_type='cumulative', change=change)).show()

## 7. Every ticker in the set

`rare_case_run` as one call. The first call runs the whole pipeline for each ticker (a few seconds
for the six synthetic tickers; longer on Yahoo, which downloads each); later calls are cached. A
ticker that fails is listed under `failed` instead of failing the request.

In [ ]:
# Q: How do streaks and return distributions compare across tickers?
start = time.perf_counter()
multi = get('/api/multi-ticker', set=TICKER_SET, drill_n_days=3)
print(f"{len(multi['tickers'])} tickers in {time.perf_counter() - start:.1f} s; "
      f"failed: {[f['symbol'] for f in multi['failed']] or 'none'}")
display(table(multi['streaks']))
table(multi['distribution'])

In [ ]:
# Q: One ticker's rare events from that run (its 3-day drill table).
first = multi['tickers'][0]
print(f"{first['label']}: {first['rows']} days, {first['start'][:10]} to {first['end'][:10]}")
table(first['drill']).head(10)

## 8. Errors

A bad request gets a status code and an explanation, not a stack trace: 422 for a request that
cannot succeed, 404 for an unknown chart, 502 when the data source fails (Yahoo unreachable),
and 501 when the `stats` extra (scipy) is missing.

In [ ]:
bad_requests = {
    'unknown ticker':           lambda: post('/api/overview', {**P, 'ticker': 'NOPE'}),
    'misspelled parameter':     lambda: post('/api/overview', {**P, 'win_thresold': 0.3}),
    'holding period not built': lambda: post('/api/rare-events', P, n_days=7),
    'unknown chart':            lambda: post('/api/charts/pie', P),
    'too many resamples':       lambda: post('/api/statistics/probabilities', P, n_boot=10_000),
}
for name, request in bad_requests.items():
    try:
        request()
        print(f'{name}: no error?')
    except ApiError as exc:
        if isinstance(exc.detail, list):     # a validation error: the message and the field
            detail = f"{exc.detail[0]['msg']}: {exc.detail[0]['loc'][-1]}"
        else:
            detail = exc.detail
        print(f'{name}: {exc.status}  {detail[:110]}')

## 9. The same numbers as the package

The API computes nothing itself: it calls `src.tools.price_return` and serializes the result.
So an answer from the server equals the package's own, value for value. (The test suite checks
this for every endpoint; here is one, by hand.)

In [ ]:
from src.tools import price_return as pr

p = pr.Params(**P)
df = pr.add_rolling_stats(pr.load_price_data(p, verbose=False), p)
direct = pr.summarize_streaks(df, pr.detect_streaks(df, p), p)
via_api = table(post('/api/streaks', P)['summary'])
pd.testing.assert_frame_equal(via_api, direct, check_dtype=False)
print('✓ /api/streaks equals summarize_streaks, value for value')